In [1]:
import pandas as pd

In [2]:
import sqlite3

In [3]:
con = sqlite3.connect('olist.db')

In [7]:
tablas = ['customers', 'orders', 'order_items', 'order_payments', 'order_reviews', 'products']

In [8]:
for t in tablas:
  pd.read_csv(f'olist_{t}_dataset.csv').to_sql(t, con, if_exists='replace', index=False)
  print('Tabla cargada:', t)

Tabla cargada: customers
Tabla cargada: orders
Tabla cargada: order_items
Tabla cargada: order_payments
Tabla cargada: order_reviews
Tabla cargada: products


In [9]:
def sql(consulta):
  return pd.read_sql(consulta, con)

In [10]:
sql("""
SELECT *
FROM orders
LIMIT 5
""")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [11]:
sql("""
SELECT *
FROM customers
LIMIT 5
""")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Q1: Do customers come back?

In [14]:
sql("""
SELECT
    COUNT(DISTINCT customer_id) AS compras,
    COUNT(DISTINCT customer_unique_id) AS personas
FROM customers
""")

,compras,personas
0,99441,96096


In [16]:
sql("""
SELECT
     customer_unique_id,
     COUNT(*) AS cantidad_compras
FROM customers
GROUP BY customer_unique_id
ORDER BY cantidad_compras DESC
LIMIT 10
""")

,customer_unique_id,cantidad_compras
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6


In [20]:
sql("""
SELECT
    cantidad_compras,
    COUNT(*) AS personas
FROM  (
    SELECT customer_unique_id, COUNT(*) AS cantidad_compras
    FROM customers
    GROUP BY customer_unique_id
)
GROUP BY cantidad_compras
ORDER BY cantidad_compras
""")

,cantidad_compras,personas
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


Q2: What do customers buy?

In [21]:
sql("""
SELECT
p.product_category_name AS categoria,
COUNT(*) AS productos_vendidos,
ROUND(SUM(oi.price)) AS ventas
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY categoria
ORDER BY productos_vendidos DESC
""")

,categoria,productos_vendidos,ventas
0,cama_mesa_banho,11115,1036989.0
1,beleza_saude,9670,1258681.0
2,esporte_lazer,8641,988049.0
3,moveis_decoracao,8334,729762.0
4,informatica_acessorios,7827,911954.0
...,...,...,...
69,la_cuisine,14,2055.0
70,cds_dvds_musicais,14,730.0
71,pc_gamer,9,1546.0
72,fashion_roupa_infanto_juvenil,8,570.0


In [22]:
sql("""
SELECT
p.product_category_name AS categoria,
COUNT(*) AS productos_vendidos,
ROUND(SUM(oi.price)) AS ventas
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY categoria
ORDER BY ventas DESC
""")

,categoria,productos_vendidos,ventas
0,beleza_saude,9670,1258681.0
1,relogios_presentes,5991,1205006.0
2,cama_mesa_banho,11115,1036989.0
3,esporte_lazer,8641,988049.0
4,informatica_acessorios,7827,911954.0
...,...,...,...
69,flores,33,1110.0
70,casa_conforto_2,30,760.0
71,cds_dvds_musicais,14,730.0
72,fashion_roupa_infanto_juvenil,8,570.0


Q3: Are late deliveries hurting us?

In [23]:
sql("""
SELECT
CASE
WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'Tarde'
ELSE 'A tiempo'
END AS entrega,
COUNT(*) AS pedidos,
ROUND(AVG(r.review_score), 2) AS puntaje_promedio
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
GROUP BY entrega
""")


,entrega,pedidos,puntaje_promedio
0,A tiempo,91523,4.21
1,Tarde,7701,2.57


In [26]:
sql("""
SELECT
CASE
WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'Tarde'
ELSE 'A tiempo'
END AS entrega,
COUNT(*) AS pedidos,
ROUND(AVG(r.review_score), 2) AS puntaje_promedio,
ROUND(100.0 * SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_resenas_malas
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
AND o.order_delivered_customer_date IS NOT NULL
GROUP BY entrega
""")

,entrega,pedidos,puntaje_promedio,pct_resenas_malas
0,A tiempo,88653,4.29,9.2
1,Tarde,7700,2.57,54.0


Q4: Who are our best costumers?

In [28]:
sql("""
SELECT
c.customer_unique_id,
COUNT(DISTINCT o.order_id) AS compras,
ROUND(SUM(p.payment_value)) AS gasto_total
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
ORDER BY gasto_total DESC
LIMIT 10
""")

,customer_unique_id,compras,gasto_total
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.0
1,da122df9eeddfedc1dc1f5349a1a690c,2,7572.0
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7275.0
3,dc4802a71eae9be1dd28f5d788ceb526,1,6929.0
4,459bef486812aa25204be022145caa62,1,6922.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6727.0
6,4007669dec559734d6f53e029e360987,1,6082.0
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.0
8,48e1ac109decbb87765a3eade6854098,1,4682.0
9,c8460e4251689ba205045f3ea17884a1,4,4656.0


In [30]:
sql("""
SELECT
CASE WHEN compras >=2 THEN 'Recurrente' ELSE 'Una sola compra' END AS tipo_cliente,
COUNT(*) AS clientes,
ROUND(AVG(gasto_total)) AS gasto_promedio
FROM (
SELECT
c.customer_unique_id,
COUNT(DISTINCT o.order_id) AS compras,
SUM(p.payment_value) AS gasto_total
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
)
GROUP BY tipo_cliente
""")

,tipo_cliente,clientes,gasto_promedio
0,Recurrente,2801,309.0
1,Una sola compra,90556,161.0
